In [ ]:
##

In [2]:
pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd

file = "/Users/avavannortwick/Desktop/Classes/DataViz/Final/MonthlyState-SNAP/FY20.xlsx"
xls = pd.ExcelFile(file)

print(xls.sheet_names)

['NERO', 'MARO', 'SERO', 'MWRO', 'SWRO', 'MPRO', 'WRO', 'US Summary']


In [2]:
df = pd.read_excel(file, sheet_name="NERO")
print(df.head(10))

  National Data Bank Version 8.2 PUBLIC - Supplemental Nutrition Assistance Program  \
0  SNAP Monthly State Participation and Benefit S...                                  
1                                                                                     
2                                                NaN                                  
3                                                                                     
4                              Fiscal Year and Month                                  
5                                                NaN                                  
6                                        Connecticut                                  
7                                           Oct 2019                                  
8                                           Nov 2019                                  
9                                           Dec 2019                                  

         Unnamed: 1 Unnamed: 2 Unnamed: 3 

In [20]:
dfs = []

for sheet in xls.sheet_names:
    if "us summary" in sheet.lower().strip():
        continue

    df = pd.read_excel(file, sheet_name=sheet, skiprows=7)
    df["region"] = sheet
    dfs.append(df)

raw_df = pd.concat(dfs, ignore_index=True)

print("\n--- SHEET ---", sheet)
print(df.head(5))
print(df.columns)


--- SHEET --- US Summary
     Alaska Unnamed: 1 Unnamed: 2 Unnamed: 3 Unnamed: 4 Unnamed: 5 region
0  Oct 2019      33787      73130   12769646   377.9455   174.6157    WRO
1  Nov 2019      37206      81320   14052128   377.6845   172.8004    WRO
2  Dec 2019      37124      81225   14007364   377.3129   172.4514    WRO
3  Jan 2020      35508      79830   13645280   384.2875   170.9292    WRO
4  Feb 2020      35286      80157   13877691   393.2917   173.1314    WRO
Index(['Alaska', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4',
       'Unnamed: 5', 'region'],
      dtype='object')


In [21]:
for sheet in xls.sheet_names:
    if "us summary" in sheet.lower():
        continue

    df = pd.read_excel(file, sheet_name=sheet, skiprows=7)

    print("\n---", sheet, "---")
    print(df.head(2))
    print(df.columns)


--- NERO ---
  Connecticut  Unnamed: 1  Unnamed: 2  Unnamed: 3  Unnamed: 4  Unnamed: 5
0    Oct 2019    213305.0    364659.0  48880678.0    229.1586    134.0449
1    Nov 2019    212893.0    363535.0  48574247.0    228.1627    133.6164
Index(['Connecticut', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4',
       'Unnamed: 5'],
      dtype='object')

--- MARO ---
   Delaware Unnamed: 1 Unnamed: 2 Unnamed: 3 Unnamed: 4 Unnamed: 5
0  Oct 2019      59858     121987   14271107    238.416   116.9888
1  Nov 2019      58485     118796   13787845   235.7501   116.0632
Index(['Delaware', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4',
       'Unnamed: 5'],
      dtype='object')

--- SERO ---
    Alabama  Unnamed: 1  Unnamed: 2  Unnamed: 3  Unnamed: 4  Unnamed: 5
0  Oct 2019    341033.0    718792.0  86304803.0    253.0688    120.0692
1  Nov 2019    339535.0    716466.0  85899443.0    252.9914    119.8933
Index(['Alabama', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4',
    

In [22]:
dfs = []

for sheet in xls.sheet_names:

    if "us summary" in sheet.lower():
        continue

    df = pd.read_excel(file, sheet_name=sheet, skiprows=7)

    df = df.dropna(how="all")

    # remove total rows
    df = df[~df.iloc[:, 0].astype(str).str.contains("Total", na=False)]

    # remove fully blank rows
    df = df[df.iloc[:, 0].notna()]

    # STEP 1: create STATE column from first column
    df = df.rename(columns={df.columns[0]: "state"})

    # STEP 2: forward fill state names
    df["state"] = df["state"].ffill()

    # rename remaining columns correctly
    df.columns = ["state", "month", "households", "persons", "x1", "x2"]

    # keep only what matters
    df = df[["state", "month", "households", "persons"]]

    # add region
    df["region"] = sheet

    dfs.append(df)

raw_df = pd.concat(dfs, ignore_index=True)

In [23]:
raw_df["state"].value_counts().head(10)

state
Oct 2019    63
Dec 2019    63
Jan 2020    63
Feb 2020    63
Mar 2020    63
Apr 2020    63
May 2020    63
Jun 2020    63
Jul 2020    63
Aug 2020    63
Name: count, dtype: int64

In [24]:
raw_df.head(20)

,state,month,households,persons,region
0,Oct 2019,213305.0,364659.0,48880678.0,NERO
1,Nov 2019,212893.0,363535.0,48574247.0,NERO
2,Dec 2019,211588.0,360523.0,48051106.0,NERO
3,Jan 2020,212256.0,360832.0,47340814.0,NERO
4,Feb 2020,212069.0,360031.0,47886445.0,NERO
5,Mar 2020,211990.0,360128.0,47814044.0,NERO
6,Apr 2020,222008.0,379936.0,83028443.0,NERO
7,May 2020,225676.0,387217.0,68812528.0,NERO
8,Jun 2020,225622.0,387097.0,68725197.0,NERO
9,Jul 2020,225789.0,386966.0,68416807.0,NERO


In [32]:
dfs = []

for sheet in xls.sheet_names:

    if "us summary" in sheet.lower():
        continue

    df = pd.read_excel(file, sheet_name=sheet, skiprows=7)

    df = df.dropna(how="all")

     # remove total rows
    df = df[~df.iloc[:, 0].astype(str).str.contains("Total", na=False)]

    # rename first column (state/month column)
    df = df.rename(columns={df.columns[0]: "state_raw"})

    # 🔥 NOW we must rename numeric columns BEFORE using them
    df.columns = ["state_raw", "households", "persons", "x1", "x2", "x3"]

    # detect state header rows (NOW this works)
    is_state_row = df["households"].isna() & df["persons"].isna()

    # assign state name
    df.loc[is_state_row, "state"] = df.loc[is_state_row, "state_raw"]

    # 🔥 FIX: manually assign FIRST state (Connecticut)
    first_state = df.loc[~is_state_row, "state_raw"].iloc[0]
    df["state"] = df["state"].fillna(first_state)

    # forward fill state
    df["state"] = df["state"].ffill()

    # remove separator rows
    df = df[~is_state_row]

    # keep only what you need
    df = df[["state", "state_raw", "households", "persons"]]

    df = df.rename(columns={"state_raw": "month"})

    df["region"] = sheet

    dfs.append(df)

raw_df = pd.concat(dfs, ignore_index=True)

In [33]:
raw_df.head(20)

,state,month,households,persons,region
0,Oct 2019,Oct 2019,213305.0,364659.0,NERO
1,Oct 2019,Nov 2019,212893.0,363535.0,NERO
2,Oct 2019,Dec 2019,211588.0,360523.0,NERO
3,Oct 2019,Jan 2020,212256.0,360832.0,NERO
4,Oct 2019,Feb 2020,212069.0,360031.0,NERO
5,Oct 2019,Mar 2020,211990.0,360128.0,NERO
6,Oct 2019,Apr 2020,222008.0,379936.0,NERO
7,Oct 2019,May 2020,225676.0,387217.0,NERO
8,Oct 2019,Jun 2020,225622.0,387097.0,NERO
9,Oct 2019,Jul 2020,225789.0,386966.0,NERO


In [36]:
dfs = []

for sheet in xls.sheet_names:

    if "us summary" in sheet.lower():
        continue

    df = pd.read_excel(file, sheet_name=sheet, skiprows=7)

    df = df.dropna(how="all")

    # remove totals
    df = df[~df.iloc[:, 0].astype(str).str.contains("Total", na=False)]

    # rename columns FIRST (important!)
    df.columns = ["state_raw", "month", "households", "persons", "x1", "x2"]

    # detect state rows (ONLY rows where numeric data is missing)
    is_state_row = df["households"].isna() & df["persons"].isna()

    # assign state only where it is actually a state
    df.loc[is_state_row, "state"] = df.loc[is_state_row, "state_raw"]

    # 🔥 MANUAL FIX for first block (Connecticut)
    df.loc[0:11, "state"] = "Connecticut"

    # forward fill state correctly
    df["state"] = df["state"].ffill()

    # remove state-only rows
    df = df[~is_state_row]

    # keep clean structure
    df = df[["state", "month", "households", "persons"]]

    df["region"] = sheet

    dfs.append(df)

raw_df = pd.concat(dfs, ignore_index=True)

In [37]:
raw_df.head(20)

,state,month,households,persons,region
0,Connecticut,213305.0,364659.0,48880678.0,NERO
1,Connecticut,212893.0,363535.0,48574247.0,NERO
2,Connecticut,211588.0,360523.0,48051106.0,NERO
3,Connecticut,212256.0,360832.0,47340814.0,NERO
4,Connecticut,212069.0,360031.0,47886445.0,NERO
5,Connecticut,211990.0,360128.0,47814044.0,NERO
6,Connecticut,222008.0,379936.0,83028443.0,NERO
7,Connecticut,225676.0,387217.0,68812528.0,NERO
8,Connecticut,225622.0,387097.0,68725197.0,NERO
9,Connecticut,225789.0,386966.0,68416807.0,NERO


In [44]:
dfs = []

for sheet in xls.sheet_names:

    if "us summary" in sheet.lower():
        continue

    df = pd.read_excel(file, sheet_name=sheet, skiprows=3)

    df = df.dropna(how="all")

    # remove top 3 junk rows 
    df = df.iloc[3:].reset_index(drop=True)

    # 🔥 FIX: keep only needed columns
    df = df.iloc[:, :3]

    # now safe to rename
    df.columns = ["col0", "households", "persons"]

    state = None
    cleaned_rows = []

    for _, row in df.iterrows():

        col0 = str(row["col0"]).strip()

        # detect state row
        if pd.isna(row["households"]) and pd.isna(row["persons"]) and col0 not in ["Total", "Fiscal Year and Month"]:
            state = col0
            continue

        # skip totals
        if col0 == "Total":
            continue

        cleaned_rows.append([
            state,
            col0,
            row["households"],
            row["persons"]
        ])

    clean_df = pd.DataFrame(cleaned_rows, columns=["state", "month", "households", "persons"])

    clean_df["region"] = sheet

    dfs.append(clean_df)

raw_df = pd.concat(dfs, ignore_index=True)

In [45]:
raw_df.head(20)

,state,month,households,persons,region
0,Connecticut,Oct 2019,213305,364659,NERO
1,Connecticut,Nov 2019,212893,363535,NERO
2,Connecticut,Dec 2019,211588,360523,NERO
3,Connecticut,Jan 2020,212256,360832,NERO
4,Connecticut,Feb 2020,212069,360031,NERO
5,Connecticut,Mar 2020,211990,360128,NERO
6,Connecticut,Apr 2020,222008,379936,NERO
7,Connecticut,May 2020,225676,387217,NERO
8,Connecticut,Jun 2020,225622,387097,NERO
9,Connecticut,Jul 2020,225789,386966,NERO


In [46]:
raw_df.tail(20)

,state,month,households,persons,region
736,Commonwealth of Northern Mariana Islands,Feb 2020,--,--,WRO
737,Commonwealth of Northern Mariana Islands,Mar 2020,--,--,WRO
738,Commonwealth of Northern Mariana Islands,Apr 2020,--,--,WRO
739,Commonwealth of Northern Mariana Islands,May 2020,--,--,WRO
740,Commonwealth of Northern Mariana Islands,Jun 2020,--,--,WRO
741,Commonwealth of Northern Mariana Islands,Jul 2020,--,--,WRO
742,Commonwealth of Northern Mariana Islands,Aug 2020,--,--,WRO
743,Commonwealth of Northern Mariana Islands,Sep 2020,--,--,WRO
744,WRO,Oct 2019,3375336,6260755,WRO
745,WRO,Nov 2019,3382224,6259426,WRO


In [47]:
raw_df.shape

(756, 5)

In [49]:
raw_df["state"].unique()

array(['Connecticut', 'Maine', 'Massachusetts', 'New Hampshire',
       'New York', 'Rhode Island', 'Vermont', 'Virgin Islands', 'NERO',
       'Delaware', 'District of Columbia', 'Maryland', 'New Jersey',
       'Pennsylvania', 'Puerto Rico', 'Virginia', 'West Virginia', 'MARO',
       'Alabama', 'Florida', 'Georgia', 'Kentucky', 'Mississippi',
       'North Carolina', 'South Carolina', 'Tennessee', 'SERO',
       'Illinois', 'Indiana', 'Iowa', 'Michigan', 'Minnesota', 'Ohio',
       'Wisconsin', 'MWRO', 'Arizona', 'Arkansas', 'Louisiana',
       'New Mexico', 'Oklahoma', 'Texas', 'Utah', 'SWRO', 'Colorado',
       'Kansas', 'Missouri', 'Montana', 'Nebraska', 'North Dakota',
       'South Dakota', 'Wyoming', 'MPRO', 'Alaska', 'American Samoa',
       'California', 'Guam', 'Hawaii', 'Idaho', 'Nevada', 'Oregon',
       'Washington', 'Commonwealth of Northern Mariana Islands', 'WRO'],
      dtype=object)

In [52]:
raw_df["state"].value_counts().head(20)

state
Connecticut     12
Nebraska        12
MWRO            12
Arizona         12
Arkansas        12
Louisiana       12
New Mexico      12
Oklahoma        12
Texas           12
Utah            12
SWRO            12
Colorado        12
Kansas          12
Missouri        12
Montana         12
North Dakota    12
Ohio            12
South Dakota    12
Wyoming         12
MPRO            12
Name: count, dtype: int64

In [53]:
# Remove rows totaling regions
region_codes = ["NERO", "MARO", "SERO", "MWRO", "SWRO", "MPRO", "WRO"]
raw_df = raw_df[~raw_df["state"].isin(region_codes)]
raw_df = raw_df[~raw_df["state"].str.contains("RO$", na=False)]
raw_df["state"].unique()

array(['Connecticut', 'Maine', 'Massachusetts', 'New Hampshire',
       'New York', 'Rhode Island', 'Vermont', 'Virgin Islands',
       'Delaware', 'District of Columbia', 'Maryland', 'New Jersey',
       'Pennsylvania', 'Puerto Rico', 'Virginia', 'West Virginia',
       'Alabama', 'Florida', 'Georgia', 'Kentucky', 'Mississippi',
       'North Carolina', 'South Carolina', 'Tennessee', 'Illinois',
       'Indiana', 'Iowa', 'Michigan', 'Minnesota', 'Ohio', 'Wisconsin',
       'Arizona', 'Arkansas', 'Louisiana', 'New Mexico', 'Oklahoma',
       'Texas', 'Utah', 'Colorado', 'Kansas', 'Missouri', 'Montana',
       'Nebraska', 'North Dakota', 'South Dakota', 'Wyoming', 'Alaska',
       'American Samoa', 'California', 'Guam', 'Hawaii', 'Idaho',
       'Nevada', 'Oregon', 'Washington',
       'Commonwealth of Northern Mariana Islands'], dtype=object)

In [54]:
keep_states = [
    "Alabama","Alaska","Arizona","Arkansas","California","Colorado","Connecticut",
    "Delaware","Florida","Georgia","Hawaii","Idaho","Illinois","Indiana","Iowa",
    "Kansas","Kentucky","Louisiana","Maine","Maryland","Massachusetts","Michigan",
    "Minnesota","Mississippi","Missouri","Montana","Nebraska","Nevada","New Hampshire",
    "New Jersey","New Mexico","New York","North Carolina","North Dakota","Ohio",
    "Oklahoma","Oregon","Pennsylvania","Rhode Island","South Carolina","South Dakota",
    "Tennessee","Texas","Utah","Vermont","Virginia","Washington","West Virginia",
    "Wisconsin","Wyoming","District of Columbia"
]

raw_df = raw_df[raw_df["state"].isin(keep_states)]

In [90]:
raw_df["year"] = 2020
df_2020 = raw_df
df_2020.head(20)

,state,month,households,persons,region,year
0,Connecticut,Oct 2019,213305,364659,NERO,2020
1,Connecticut,Nov 2019,212893,363535,NERO,2020
2,Connecticut,Dec 2019,211588,360523,NERO,2020
3,Connecticut,Jan 2020,212256,360832,NERO,2020
4,Connecticut,Feb 2020,212069,360031,NERO,2020
5,Connecticut,Mar 2020,211990,360128,NERO,2020
6,Connecticut,Apr 2020,222008,379936,NERO,2020
7,Connecticut,May 2020,225676,387217,NERO,2020
8,Connecticut,Jun 2020,225622,387097,NERO,2020
9,Connecticut,Jul 2020,225789,386966,NERO,2020


## Repeating for each year

In [78]:
import pandas as pd
import re

def clean_snap_file(file_path, xls):

    dfs = []

    # 🔴 STEP A: automatically extract year from filename
    year = re.search(r"FY(\d{2})", file_path).group(1)
    year = "20" + year   # FY20 → 2020

    for sheet in xls.sheet_names:

        if "us summary" in sheet.lower():
            continue

        df = pd.read_excel(file_path, sheet_name=sheet, skiprows=3)

        # remove top 3 junk rows  (YOU WANTED TO KEEP THIS)
        df = df.iloc[3:].reset_index(drop=True)

        df = df.dropna(how="all")

        df = df.iloc[:, :3]

        df.columns = ["col0", "households", "persons"]

        state = None
        cleaned_rows = []

        for _, row in df.iterrows():

            col0 = str(row["col0"]).strip()

            # detect state header rows
            if pd.isna(row["households"]) and pd.isna(row["persons"]) and col0 != "Total":
                state = col0
                continue

            # skip totals
            if col0 == "Total":
                continue

            cleaned_rows.append([
                state,
                col0,
                row["households"],
                row["persons"],
                sheet,
                year
            ])

        clean_df = pd.DataFrame(
            cleaned_rows,
            columns=["state", "month", "households", "persons", "region", "year"]
        )

        dfs.append(clean_df)

    raw_df = pd.concat(dfs, ignore_index=True)

    # 🔥 KEEP ONLY 50 STATES + DC
    keep_states = [
        "Alabama","Alaska","Arizona","Arkansas","California","Colorado","Connecticut",
        "Delaware","Florida","Georgia","Hawaii","Idaho","Illinois","Indiana","Iowa",
        "Kansas","Kentucky","Louisiana","Maine","Maryland","Massachusetts","Michigan",
        "Minnesota","Mississippi","Missouri","Montana","Nebraska","Nevada","New Hampshire",
        "New Jersey","New Mexico","New York","North Carolina","North Dakota","Ohio",
        "Oklahoma","Oregon","Pennsylvania","Rhode Island","South Carolina","South Dakota",
        "Tennessee","Texas","Utah","Vermont","Virginia","Washington","West Virginia",
        "Wisconsin","Wyoming","District of Columbia"
    ]

    raw_df = raw_df[raw_df["state"].isin(keep_states)]

    return raw_df

In [79]:
file = "/Users/avavannortwick/Desktop/Classes/DataViz/Final/MonthlyState-SNAP/FY21.xlsx"   # 🔴 ONLY CHANGE THIS LINE
xls = pd.ExcelFile(file)
df_2021 = clean_snap_file(file, xls)

In [80]:
df_2021.head()

,state,month,households,persons,region,year
0,Connecticut,Oct 2020,213541,365607,NERO,2021
1,Connecticut,Nov 2020,210279,360157,NERO,2021
2,Connecticut,Dec 2020,213589,365498,NERO,2021
3,Connecticut,Jan 2021,215623,368684,NERO,2021
4,Connecticut,Feb 2021,218820,373747,NERO,2021


In [81]:
file = "/Users/avavannortwick/Desktop/Classes/DataViz/Final/MonthlyState-SNAP/FY22.xlsx"
xls = pd.ExcelFile(file)
df_2022 = clean_snap_file(file, xls)
df_2022.head()

,state,month,households,persons,region,year
0,Connecticut,Oct 2021,216821,369252,NERO,2022
1,Connecticut,Nov 2021,214609,365484,NERO,2022
2,Connecticut,Dec 2021,215995,368265,NERO,2022
3,Connecticut,Jan 2022,215898,369521,NERO,2022
4,Connecticut,Feb 2022,217360,373068,NERO,2022


In [82]:
file = "/Users/avavannortwick/Desktop/Classes/DataViz/Final/MonthlyState-SNAP/FY23.xlsx"
xls = pd.ExcelFile(file)
df_2023 = clean_snap_file(file, xls)
df_2023.head()

,state,month,households,persons,region,year
0,Connecticut,Oct 2022,224381,387527,NERO,2023
1,Connecticut,Nov 2022,224284,385500,NERO,2023
2,Connecticut,Dec 2022,225221,387126,NERO,2023
3,Connecticut,Jan 2023,225538,387971,NERO,2023
4,Connecticut,Feb 2023,226968,390726,NERO,2023


In [83]:
file = "/Users/avavannortwick/Desktop/Classes/DataViz/Final/MonthlyState-SNAP/FY24.xlsx"
xls = pd.ExcelFile(file)
df_2024 = clean_snap_file(file, xls)
df_2024.head()

,state,month,households,persons,region,year
0,Connecticut,Oct 2023,229750,393847,NERO,2024
1,Connecticut,Nov 2023,228992,392246,NERO,2024
2,Connecticut,Dec 2023,227888,389776,NERO,2024
3,Connecticut,Jan 2024,227629,387807,NERO,2024
4,Connecticut,Feb 2024,228138,388350,NERO,2024


In [84]:
file = "/Users/avavannortwick/Desktop/Classes/DataViz/Final/MonthlyState-SNAP/FY25.xlsx"
xls = pd.ExcelFile(file)
df_2025 = clean_snap_file(file, xls)
df_2025.head()

,state,month,households,persons,region,year
0,Connecticut,Oct 2024,230274,390261,NERO,2025
1,Connecticut,Nov 2024,228462,386771,NERO,2025
2,Connecticut,Dec 2024,225536,381151,NERO,2025
3,Connecticut,Jan 2025,224287,377990,NERO,2025
4,Connecticut,Feb 2025,220667,371398,NERO,2025


In [85]:
file = "/Users/avavannortwick/Desktop/Classes/DataViz/Final/MonthlyState-SNAP/FY26.xlsx"
xls = pd.ExcelFile(file)
df_2026 = clean_snap_file(file, xls)
df_2026.head()

,state,month,households,persons,region,year
0,Connecticut,Oct 2025,209322,348163,NERO,2026
1,Connecticut,Nov 2025,204188,338147,NERO,2026
2,Connecticut,Dec 2025,--,--,NERO,2026
3,Connecticut,Jan 2026,--,--,NERO,2026
4,Connecticut,Feb 2026,--,--,NERO,2026


In [87]:
df_2021.shape


(612, 6)

In [91]:
df_2020.to_csv("snap_2020_clean.csv", index=False)
df_2021.to_csv("snap_2021_clean.csv", index=False)
df_2022.to_csv("snap_2022_clean.csv", index=False)
df_2023.to_csv("snap_2023_clean.csv", index=False)
df_2024.to_csv("snap_2024_clean.csv", index=False)
df_2025.to_csv("snap_2025_clean.csv", index=False)
df_2026.to_csv("snap_2026_clean.csv", index=False)

## Make into one dataset

In [92]:
whole_df = pd.concat([
    df_2020,
    df_2021,
    df_2022,
    df_2023,
    df_2024,
    df_2025,
    df_2026
], ignore_index=True)

print(whole_df.shape)
print(whole_df["year"].unique())
print(whole_df["state"].nunique())

(4284, 6)
[2020 '2021' '2022' '2023' '2024' '2025' '2026']
51


In [95]:
whole_df["date"] = pd.to_datetime(whole_df["month"], format="%b %Y")
whole_df = whole_df.sort_values(["state", "date"])
whole_df.to_csv("snap_panel_2020_2026.csv", index=False)
whole_df.head(20)

,state,month,households,persons,region,year,date
168,Alabama,Oct 2019,341033,718792,SERO,2020,2019-10-01
169,Alabama,Nov 2019,339535,716466,SERO,2020,2019-11-01
170,Alabama,Dec 2019,337577,711384,SERO,2020,2019-12-01
171,Alabama,Jan 2020,335156,705542,SERO,2020,2020-01-01
172,Alabama,Feb 2020,334581,704788,SERO,2020,2020-02-01
173,Alabama,Mar 2020,335919,706626,SERO,2020,2020-03-01
174,Alabama,Apr 2020,354410,739950,SERO,2020,2020-04-01
175,Alabama,May 2020,363723,755038,SERO,2020,2020-05-01
176,Alabama,Jun 2020,365109,755694,SERO,2020,2020-06-01
177,Alabama,Jul 2020,361924,746774,SERO,2020,2020-07-01
